In [ ]:
#This is the only one code box that need to be edited when adding new companies to the dictionaries.

# Add new comp info for yield dictionary, e.g.:
#new_yield_comp = {
#    'company_name': 'Fortinet',
#    'company_CIK': '1262039',
#    'cash_and_CE': 'CashAndCashEquivalentsAtCarryingValue',
#    'short_term_investments': 'OtherShortTermInvestments+EquitySecuritiesFvNi',
#    'long_term_investments': 'LongTermInvestments',
#    'interest_income': 'InvestmentIncomeNet'
#}

# If not add yield comp, just leave the item empty (e.g. 'company_name': '')

new_yield_comp = {
    'company_name': 'Fortinet',
    'company_CIK': '1262039',
    'cash_and_CE': 'CashAndCashEquivalentsAtCarryingValue',
    'short_term_investments': 'OtherShortTermInvestments+EquitySecuritiesFvNi',
    'long_term_investments': 'LongTermInvestments',
    'interest_income': 'InvestmentIncomeNet'
}


# Add new comp info for maturity dictionary, e.g.:
#new_maturity_company = {
#  'company_name': 'Vertex Pharmaceuticals',
#  'company_CIK': '0000875320',
#  'maturity_tag': '{'0.5': 'AvailableForSaleSecuritiesDebtMaturitiesWithinOneYearFairValue', \
#                    '3': 'AvailableForSaleSecuritiesDebtMaturitiesAfterOneThroughFiveYearsFairValue', \
#                    '7.5': 'AvailableForSaleSecuritiesDebtMaturitiesAfterFiveThroughTenYearsFairValue'}'
#}

# If not add maturity comp, just leave the item empty (e.g. 'company_name': '')

new_maturity_comp = {
  'company_name': 'Vertex Pharmaceuticals',
  'company_CIK': '875320',
  'maturity_tag': "{'0.5': 'AvailableForSaleSecuritiesDebtMaturitiesWithinOneYearFairValue', \
                    '3': 'AvailableForSaleSecuritiesDebtMaturitiesAfterOneThroughFiveYearsFairValue', \
                    '7.5': 'AvailableForSaleSecuritiesDebtMaturitiesAfterFiveThroughTenYearsFairValue'}"
}

In [ ]:
from google.cloud import bigquery

def insert_company_if_not_exists(project_id, dataset_id, table_id, new_company):

    client = bigquery.Client(project=project_id)
    table_ref = client.dataset(dataset_id).table(table_id)
    query = f"""
    SELECT COUNT(*) as count
    FROM `{project_id}.{dataset_id}.{table_id}`
    WHERE company_CIK = @company_CIK
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("company_CIK", "INTEGER", new_company['company_CIK'])
        ]
    )

    query_job = client.query(query, job_config=job_config)
    result = query_job.result()

    # Check if the company exists
    company_exists = False
    for row in result:
        if row.count > 0:
            company_exists = True

    if company_exists:
        print("The company already exists in the table")
    else:
        # Insert the new data if the company does not exist
        rows_to_insert = [new_company]
        errors = client.insert_rows_json(table_ref, rows_to_insert)  # API request
        if not errors:
            print("New company successfully inserted into the table")
        else:
            print(f"Error inserting data: {errors}")


In [ ]:
project_id = 'pragmatic-will-424104-n4'
dataset_id = 'PanSecDB'

if new_yield_comp['company_name'] != '':
    table_id = 'YieldDict'
    insert_company_if_not_exists(project_id, dataset_id, table_id, new_yield_comp)


if new_maturity_comp['company_name'] != '':
    table_id = 'MaturityDict'
    insert_company_if_not_exists(project_id, dataset_id, table_id, new_maturity_comp)

The company already exists in the table
The company already exists in the table
